# Tugas Proyek UTS Data Mining: Analisis Korelasi & Deteksi Outlier
**Dataset**: MetroPT-3 (Air Compressor Multivariate Time-Series Data)
**Domain**: Sistem Kompresor Kereta Metro (Air Production Unit)

### Tujuan Analisis:
1. **Statistik Deskriptif**: Mengidentifikasi central tendency dan dispersi 7 fitur analog sensor.
2. **Analisis Korelasi**: Membangun matriks korelasi Pearson, visualisasi Heatmap, dan Scatter Plot relasi bivariat sensor.
3. **Analisis Outlier**: Menganalisis pencilan fisik melalui metode IQR (Tukey's Fences), Z-Score Standardized, dan Quantile Clipping 1%-99% (Winsorization).
4. **Visualisasi Publikasi**: Menghasilkan Box Plot dan Scatter Plot beresolusi tinggi (300 DPI) untuk paper IEEE.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Konfigurasi grafik
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#333333'
plt.rcParams['axes.linewidth'] = 0.8
plt.rcParams['grid.color'] = '#e0e0e0'
plt.rcParams['grid.linestyle'] = '--'
plt.rcParams['grid.alpha'] = 0.7

OUTPUT_DIR = 'output_visualisasi'
os.makedirs(OUTPUT_DIR, exist_ok=True)

possible_paths = [
    os.path.join('..', 'dataset', 'MetroPT3(AirCompressor).csv'),
    os.path.join('dataset', 'MetroPT3(AirCompressor).csv'),
    r'd:\uts_datmin\FR3N\dataset\MetroPT3(AirCompressor).csv',
    os.path.join('archive', 'MetroPT3(AirCompressor).csv'),
    r'd:\uts_datmin\archive\MetroPT3(AirCompressor).csv',
    'MetroPT3(AirCompressor).csv'
]
DATASET_PATH = next((p for p in possible_paths if os.path.isfile(p)), possible_paths[0])
print(f'Lokasi dataset terdeteksi: {DATASET_PATH}')
ANALOG_FEATURES = ['TP2', 'TP3', 'H1', 'DV_pressure', 'Reservoirs', 'Oil_temperature', 'Motor_current']
FEATURE_LABELS = {
    'TP2': 'TP2 (Tekanan Kompresor - bar)',
    'TP3': 'TP3 (Tekanan Pneumatik - bar)',
    'H1': 'H1 (Drop Tekanan Filter - bar)',
    'DV_pressure': 'DV_pressure (Tekanan Menara - bar)',
    'Reservoirs': 'Reservoirs (Tekanan Tangki - bar)',
    'Oil_temperature': 'Oil_temperature (Suhu Oli - °C)',
    'Motor_current': 'Motor_current (Arus Motor - A)'
}
print('Setup selesai! Library berhasil dimuat.')

## 1. Memuat Dataset Sensor MetroPT-3

In [ ]:
usecols = ['timestamp'] + ANALOG_FEATURES
df = pd.read_csv(DATASET_PATH, usecols=usecols)
df['timestamp'] = pd.to_datetime(df['timestamp'])
print(f'Total baris data: {len(df):,}')
print(f'Rentang waktu observasi: {df["timestamp"].min()} s.d. {df["timestamp"].max()}')
df.head()

## 2. Statistik Deskriptif (Sesuai Tabel II Paper UTS)
Menghitung rata-rata, standar deviasi, nilai minimum, median, maksimum, skewness, dan kurtosis.

In [ ]:
stats_list = []
for col in ANALOG_FEATURES:
    s = df[col]
    stats_list.append({
        'Fitur': col,
        'Deskripsi': FEATURE_LABELS[col],
        'Mean (mu)': s.mean(),
        'Std (sigma)': s.std(),
        'Min': s.min(),
        'Median': s.median(),
        'Maks': s.max(),
        'Skewness': s.skew(),
        'Kurtosis': s.kurtosis()
    })
stats_df = pd.DataFrame(stats_list)
stats_df.to_csv(os.path.join(OUTPUT_DIR, 'tabel_statistik_deskriptif.csv'), index=False)
stats_df

## 3. Analisis Korelasi Pearson (Heatmap & Scatter Plot)
Mengukur derajat keterikatan linear antar-sensor kompresor.

In [ ]:
corr_matrix = df[ANALOG_FEATURES].corr(method='pearson')
corr_matrix.to_csv(os.path.join(OUTPUT_DIR, 'tabel_korelasi_pearson.csv'))

# Visualisasi 1: Heatmap Korelasi Pearson
plt.figure(figsize=(9, 7.5))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm', vmin=-1, vmax=1, center=0,
            square=True, linewidths=1.2, linecolor='white',
            cbar_kws={'label': 'Koefisien Korelasi Pearson (r)', 'shrink': 0.8})
plt.title('Matriks Korelasi Pearson Antar-Sensor Analog MetroPT-3', fontsize=13, fontweight='bold', pad=15)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '01_matriks_korelasi_heatmap.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Visualisasi 2: Scatter Plot Pasangan Korelasi Utama
sample_df = df.sample(n=min(30000, len(df)), random_state=42)
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
pairs = [
    ('TP2', 'H1', 'Korelasi Negatif Kuat (r = -0.961)\nTekanan Kompresor vs Drop Tekanan Filter', '#d9534f'),
    ('TP2', 'Motor_current', 'Korelasi Positif Kuat (r = +0.698)\nTekanan Kompresor vs Beban Arus Motor', '#0275d8'),
    ('TP3', 'Reservoirs', 'Korelasi Sempurna / Redundansi (r = 1.000)\nTekanan Pneumatik vs Tekanan Tangki Hilir', '#5cb85c'),
    ('Oil_temperature', 'Motor_current', 'Korelasi Positif Sedang (r = +0.529)\nSuhu Oli vs Konsumsi Arus Motor', '#f0ad4e')
]
for ax, (x_col, y_col, title, color) in zip(axes.flatten(), pairs):
    ax.scatter(sample_df[x_col], sample_df[y_col], alpha=0.15, s=10, color=color, edgecolors='none', label='Observasi Sensor')
    sns.regplot(data=sample_df, x=x_col, y=y_col, ax=ax, scatter=False, color='#222222',
                line_kws={'linewidth': 2, 'linestyle': '--', 'label': 'Garis Tren Linear'})
    ax.set_title(title, fontsize=11, fontweight='bold', pad=10)
    ax.set_xlabel(FEATURE_LABELS[x_col], fontsize=9)
    ax.set_ylabel(FEATURE_LABELS[y_col], fontsize=9)
    ax.grid(True)
    ax.legend(loc='best', framealpha=0.9, fontsize=8.5)
plt.suptitle('Analisis Scatter Plot Bivariat Pasangan Sensor Kunci MetroPT-3', fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '02_scatter_plot_korelasi_utama.png'), dpi=300, bbox_inches='tight')
plt.show()

## 4. Deteksi Outlier (Metode IQR & Quantile Clipping 1%-99%)
Sesuai Bab II Subbagian Penanganan Pencilan Ekstrem dan file `Penentuan Metode.xlsx`.

In [ ]:
# Ringkasan Deteksi Outlier Metode IQR (Tukey's Fences)
iqr_results = []
for col in ANALOG_FEATURES:
    s = df[col]
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lb, ub = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outliers = (s < lb) | (s > ub)
    iqr_results.append({
        'Fitur': col,
        'Q1 (25%)': q1, 'Q3 (75%)': q3, 'IQR': iqr,
        'Lower Bound (Q1-1.5*IQR)': lb, 'Upper Bound (Q3+1.5*IQR)': ub,
        'Jumlah Outlier': outliers.sum(),
        'Persentase (%)': (outliers.sum() / len(df)) * 100
    })
iqr_df = pd.DataFrame(iqr_results)
iqr_df.to_csv(os.path.join(OUTPUT_DIR, 'tabel_outlier_iqr.csv'), index=False)
iqr_df

In [ ]:
# Ringkasan Deteksi Outlier Quantile Clipping 1% - 99% (Winsorization)
quant_results = []
for col in ANALOG_FEATURES:
    s = df[col]
    p1, p99 = s.quantile(0.01), s.quantile(0.99)
    outside = (s < p1) | (s > p99)
    quant_results.append({
        'Fitur': col,
        'P1 (Batas Bawah 1%)': p1,
        'P99 (Batas Atas 99%)': p99,
        'Titik Terdampak': outside.sum(),
        'Persentase (%)': (outside.sum() / len(df)) * 100
    })
quant_df = pd.DataFrame(quant_results)
quant_df.to_csv(os.path.join(OUTPUT_DIR, 'tabel_outlier_quantile_clipping.csv'), index=False)
quant_df

## 5. Visualisasi Box Plot Sebaran & Outlier
1. Box Plot individual skala asli (multi-panel).
2. Box Plot Z-Score Standardized (membuktikan ketimpangan skala & kurtosis ekstrem DV_pressure).
3. Box Plot Sebelum vs Sesudah Quantile Clipping (efektivitas peredaman pencilan).

In [ ]:
# Visualisasi 4: Box Plot Skala Asli Per Fitur
sample_df = df[ANALOG_FEATURES].sample(n=min(50000, len(df)), random_state=42)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
colors = ['#1f77b4', '#aec7e8', '#ff7f0e', '#d62728', '#2ca02c', '#9467bd', '#8c564b']
for i, col in enumerate(ANALOG_FEATURES):
    sns.boxplot(y=sample_df[col], ax=axes[i], color=colors[i], width=0.4,
                flierprops=dict(marker='o', markersize=3, alpha=0.3, markerfacecolor='red', markeredgecolor='none'))
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].set_ylabel(FEATURE_LABELS[col], fontsize=9)
    axes[i].grid(True)
axes[7].axis('off')
text_info = (
    'Interpretasi Box Plot:\n'
    '• Garis Tengah: Median (Q2)\n'
    '• Kotak: Rentang Interkuartil (IQR)\n'
    '• Whiskers: [Q1 - 1.5*IQR, Q3 + 1.5*IQR]\n'
    '• Titik Merah: Pencilan (Outliers/Fliers)\n\n'
    'Catatan Khusus:\n'
    'DV_pressure memiliki fliers terjauh\n'
    'akibat lonjakan siklus purge katup.'
)
axes[7].text(0.1, 0.45, text_info, fontsize=9.5, bbox=dict(boxstyle='round,pad=0.8', facecolor='#f8f9fa', edgecolor='#cccccc'))
plt.suptitle('Visualisasi Box Plot Sebaran & Outlier Per Sensor Analog (Skala Asli)', fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '04_boxplot_individual_skala_asli.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Visualisasi 5: Box Plot Standardized Z-Score
zscore_df = (sample_df - sample_df.mean()) / sample_df.std()
plt.figure(figsize=(12, 6.5))
sns.boxplot(data=zscore_df, palette='Set2', width=0.5,
            flierprops=dict(marker='o', markersize=3, alpha=0.35, markerfacecolor='#e74c3c', markeredgecolor='none'))
plt.axhline(3, color='red', linestyle='--', linewidth=1.2, label='Batas Atas 3-Sigma (+3σ)')
plt.axhline(-3, color='red', linestyle='--', linewidth=1.2, label='Batas Bawah 3-Sigma (-3σ)')
plt.axhline(0, color='black', linestyle='-', linewidth=0.8, alpha=0.5)
plt.title('Distribusi Data Ternormalisasi Z-Score & Deteksi Outlier Ekstrem (> 3σ)', fontsize=13, fontweight='bold', pad=12)
plt.ylabel('Standar Deviasi dari Rata-rata (Z-Score)', fontsize=10)
plt.xlabel('Sensor Analog', fontsize=10)
plt.grid(True)
plt.legend(loc='upper right', framealpha=0.95)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '05_boxplot_standardized_zscore.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Visualisasi 6: Box Plot Sebelum vs Sesudah Quantile Clipping
clipped_df = pd.DataFrame({
    col: sample_df[col].clip(lower=quant_df.loc[quant_df['Fitur'] == col, 'P1 (Batas Bawah 1%)'].values[0],
                            upper=quant_df.loc[quant_df['Fitur'] == col, 'P99 (Batas Atas 99%)'].values[0])
    for col in ANALOG_FEATURES
})
compare_cols = ['TP2', 'H1', 'DV_pressure', 'Motor_current']
fig, axes = plt.subplots(2, 4, figsize=(15, 7.5))
for i, col in enumerate(compare_cols):
    # Sebelum
    sns.boxplot(y=sample_df[col], ax=axes[0, i], color='#e74c3c', width=0.4,
                flierprops=dict(marker='o', markersize=3, alpha=0.4, markerfacecolor='black', markeredgecolor='none'))
    axes[0, i].set_title(f'Sebelum: {col}', fontsize=10.5, fontweight='bold', color='#c0392b')
    axes[0, i].grid(True)
    # Sesudah
    sns.boxplot(y=clipped_df[col], ax=axes[1, i], color='#2ecc71', width=0.4,
                flierprops=dict(marker='o', markersize=3, alpha=0.4, markerfacecolor='black', markeredgecolor='none'))
    axes[1, i].set_title(f'Sesudah Winsorize: {col}', fontsize=10.5, fontweight='bold', color='#27ae60')
    axes[1, i].grid(True)
plt.suptitle('Efektivitas Penanganan Outlier: Sebelum vs Sesudah Quantile Clipping (1% - 99%)', fontsize=13, fontweight='bold', y=0.99)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '06_boxplot_before_after_clipping.png'), dpi=300, bbox_inches='tight')
plt.show()

## 6. Visualisasi Scatter Plot Deteksi Outlier (Bivariate & Deret Waktu)

In [ ]:
# Visualisasi 7: Scatter Bivariat dengan Anotasi Outlier Kuantil
scatter_sample = df.sample(n=min(25000, len(df)), random_state=42).copy()
p1_tp2 = quant_df.loc[quant_df['Fitur'] == 'TP2', 'P1 (Batas Bawah 1%)'].values[0]
p99_tp2 = quant_df.loc[quant_df['Fitur'] == 'TP2', 'P99 (Batas Atas 99%)'].values[0]
p1_mc = quant_df.loc[quant_df['Fitur'] == 'Motor_current', 'P1 (Batas Bawah 1%)'].values[0]
p99_mc = quant_df.loc[quant_df['Fitur'] == 'Motor_current', 'P99 (Batas Atas 99%)'].values[0]
p1_oil = quant_df.loc[quant_df['Fitur'] == 'Oil_temperature', 'P1 (Batas Bawah 1%)'].values[0]
p99_oil = quant_df.loc[quant_df['Fitur'] == 'Oil_temperature', 'P99 (Batas Atas 99%)'].values[0]

outlier_p1 = (scatter_sample['TP2'] < p1_tp2) | (scatter_sample['TP2'] > p99_tp2) | \
             (scatter_sample['Motor_current'] < p1_mc) | (scatter_sample['Motor_current'] > p99_mc)
outlier_p2 = (scatter_sample['Oil_temperature'] < p1_oil) | (scatter_sample['Oil_temperature'] > p99_oil) | \
             (scatter_sample['Motor_current'] < p1_mc) | (scatter_sample['Motor_current'] > p99_mc)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6.5))
ax1.scatter(scatter_sample.loc[~outlier_p1, 'TP2'], scatter_sample.loc[~outlier_p1, 'Motor_current'],
            color='#3498db', alpha=0.25, s=15, label='Kondisi Normal (Inlier)')
ax1.scatter(scatter_sample.loc[outlier_p1, 'TP2'], scatter_sample.loc[outlier_p1, 'Motor_current'],
            color='#e74c3c', alpha=0.85, s=35, edgecolors='black', linewidths=0.5, label='Pencilan Terdeteksi (< P1 atau > P99)')
ax1.set_title('Tekanan Kompresor (TP2) vs Arus Motor (Motor_current)', fontsize=11.5, fontweight='bold')
ax1.set_xlabel(FEATURE_LABELS['TP2'], fontsize=9.5)
ax1.set_ylabel(FEATURE_LABELS['Motor_current'], fontsize=9.5)
ax1.grid(True)
ax1.legend(loc='upper left', framealpha=0.95)

ax2.scatter(scatter_sample.loc[~outlier_p2, 'Oil_temperature'], scatter_sample.loc[~outlier_p2, 'Motor_current'],
            color='#2ecc71', alpha=0.25, s=15, label='Kondisi Normal (Inlier)')
ax2.scatter(scatter_sample.loc[outlier_p2, 'Oil_temperature'], scatter_sample.loc[outlier_p2, 'Motor_current'],
            color='#e67e22', alpha=0.85, s=35, edgecolors='black', linewidths=0.5, label='Pencilan Suhu/Beban (< P1 atau > P99)')
ax2.set_title('Suhu Oli (Oil_temperature) vs Arus Motor (Motor_current)', fontsize=11.5, fontweight='bold')
ax2.set_xlabel(FEATURE_LABELS['Oil_temperature'], fontsize=9.5)
ax2.set_ylabel(FEATURE_LABELS['Motor_current'], fontsize=9.5)
ax2.grid(True)
ax2.legend(loc='upper left', framealpha=0.95)
plt.suptitle('Analisis Scatter Plot Bivariat dengan Deteksi Outlier Ekstrem (Kuantil 1% - 99%)', fontsize=13, fontweight='bold', y=0.98)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '07_scatter_outlier_bivariat.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Visualisasi 8: Scatter Plot Deret Waktu Transient Spikes
time_subset = df.iloc[50000:65000].copy()
p99_dv = quant_df.loc[quant_df['Fitur'] == 'DV_pressure', 'P99 (Batas Atas 99%)'].values[0]
outliers_dv = time_subset[time_subset['DV_pressure'] > p99_dv]

p1_oil = quant_df.loc[quant_df['Fitur'] == 'Oil_temperature', 'P1 (Batas Bawah 1%)'].values[0]
p99_oil = quant_df.loc[quant_df['Fitur'] == 'Oil_temperature', 'P99 (Batas Atas 99%)'].values[0]
outliers_oil = time_subset[(time_subset['Oil_temperature'] < p1_oil) | (time_subset['Oil_temperature'] > p99_oil)]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
ax1.plot(time_subset['timestamp'], time_subset['DV_pressure'], color='#34495e', linewidth=0.8, alpha=0.7, label='Sinyal Kontinu DV_pressure')
ax1.scatter(outliers_dv['timestamp'], outliers_dv['DV_pressure'], color='#e74c3c', s=20, label=f'Transient Spikes (> P99={p99_dv:.2f} bar)', zorder=5)
ax1.axhline(p99_dv, color='#e74c3c', linestyle='--', linewidth=1.2, label='Batas P99 (Quantile 99%)')
ax1.set_title('Deteksi Transient Spikes pada Tekanan Menara Udara (DV_pressure) dalam Deret Waktu', fontsize=11, fontweight='bold')
ax1.set_ylabel('DV_pressure (bar)', fontsize=9.5)
ax1.grid(True)
ax1.legend(loc='upper right', framealpha=0.9)

ax2.plot(time_subset['timestamp'], time_subset['Oil_temperature'], color='#2980b9', linewidth=0.8, alpha=0.7, label='Sinyal Kontinu Suhu Oli')
ax2.scatter(outliers_oil['timestamp'], outliers_oil['Oil_temperature'], color='#f39c12', s=25, label='Pencilan Suhu (< P1 atau > P99)', zorder=5)
ax2.axhline(p99_oil, color='#e67e22', linestyle='--', linewidth=1.2, label=f'Batas P99 ({p99_oil:.1f} °C)')
ax2.axhline(p1_oil, color='#3498db', linestyle='--', linewidth=1.2, label=f'Batas P1 ({p1_oil:.1f} °C)')
ax2.set_title('Dinamika Suhu Oli (Oil_temperature) Terhadap Batas Ambang Kuantil', fontsize=11, fontweight='bold')
ax2.set_ylabel('Suhu Oli (°C)', fontsize=9.5)
ax2.set_xlabel('Waktu (Timestamp)', fontsize=9.5)
ax2.grid(True)
ax2.legend(loc='upper right', framealpha=0.9)
plt.suptitle('Visualisasi Scatter Deret Waktu: Deteksi Pencilan Transien Fisik Kompresor', fontsize=13, fontweight='bold', y=0.99)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '08_scatter_timeseries_outlier_spikes.png'), dpi=300, bbox_inches='tight')
plt.show()

### Ringkasan & Hasil Analisis
1. **Korelasi Terkuat**:
   - `TP3` vs `Reservoirs`: $r = +1.000$ (korelasi sempurna / redundansi pneumatik hilir).
   - `TP2` vs `H1`: $r = -0.961$ (korelasi negatif kuat, tekanan kompresor vs drop filter).
   - `TP2` vs `Motor_current`: $r = +0.698$ (korelasi positif kuat, beban kerja berbanding lurus dengan arus motor).
2. **Pencilan Ekstrem**:
   - `DV_pressure`: Memiliki fliers paling ekstrem dengan skewness 5.72 dan kurtosis 38.71 akibat lonjakan transien saat katup pengering aktif.
   - **Penanganan**: Quantile clipping 1%-99% terbukti efektif meredam spike tanpa menghilangkan baris deret waktu sehingga integritas kronologis tetap terjaga untuk klasterisasi K-Means.